In [1]:
import os
import pandas as pd
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [83]:
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
original_files = home + "Other/"
downloads_ncbi = home + "NCBI_Virus/downloads/11-01-2021--08-15-2025_Antarctica_North_America_South_America/"
h5n1_files = home + "Combinations/Andersen_NCBI_Virus_GISAID/11-01-2021--07-25-2025_Antarctica_North_America_South_America/" # 11-01-2021--08-15-2025_Antarctica_North_America_South_America/"
# h5n1_files = home + "GISAID/complete/2021-11-01--2025-08-15_Antarctica_North_America_South_America/"
# h5n1_files = home + "Combinations/GISAID_Andersen_NCBI_Virus/11-01-2021--06-13-2025_Antarctica_North_America_South_America/updated_06-27-2025/"
# references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
os.chdir(downloads_ncbi)
accessions = pd.read_csv("sequences.csv")

original_file_fastas = {}
for dirpath, dirs, files in os.walk(original_files):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".txt" in file_name:
            # print(file_name)
            fasta_file = df_from_fasta(file_name) # Convert fasta file to dataframe
            # print(fasta_file)
            isolates = fasta_file["full_header"].apply(lambda x: x.split("/")[3])
            header0 = fasta_file["full_header"].apply(lambda x: x.split(".")[0] + "|")
            header1 = fasta_file["full_header"].apply(lambda x: x.split(")")[0] + ")") 
            header2 = header1.apply(lambda x: x.replace(x.split("(")[0], ""))
            fasta_file["full_header"] = header0 + header2
            fasta_file["full_header"] = fasta_file["full_header"].apply(lambda x: x.replace("(", "", 1))
            # print(fasta_file["full_header"])
            fasta_file["isolate"] = isolates
            fasta_file["Accession"] = fasta_file["full_header"].apply(lambda x: x.split("|")[0][1:].strip())
            # print(isolates)
            original_file_fastas[file_name] = fasta_file

h5n1_file_fastas = {}
for dirpath, dirs, files in os.walk(h5n1_files):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        if ".fasta" in file_name:
            fasta_file = df_from_fasta(file_name) # Convert fasta file to dataframe
            # print(fasta_file)
            print(file_name)
            try:
                isolates = fasta_file["full_header"].apply(lambda x: x.split("/")[3])
                fasta_file["isolate"] = isolates
                # sra_accessions = fasta_file["full_header"].apply(lambda x: x.split("|")[0][1:])
                # fasta_file["SRA_Accession"] = sra_accessions
                h5n1_file_fastas[file_name] = fasta_file
            except:
                print("Failed", file_name)
                continue

count1 = 0
for key in original_file_fastas:
    count2 = 0
    og_fasta = original_file_fastas[key]
    # print(og_fasta)
    for h5n1_key in h5n1_file_fastas:
        h5n1_fasta = h5n1_file_fastas[h5n1_key]
        # metadata = accessions.merge(h5n1_fasta, on="SRA_Accession", how="left")
        # metadata["Accession"] = metadata["Accession"].apply(lambda x: x.split(".")[0].strip()) # Remove version
        # print(metadata[metadata["Accession"] == "PQ719222"])
        # common = metadata
        common = og_fasta.merge(h5n1_fasta, on=["isolate"], how="left")
        # print(common)
        if len(common["full_header_y"].dropna()) > 0:
            # print(common)
            # common["full_header"] = common["full_header_y"]
            # common["sequence"] = common["sequence_x"]
            # common = common[["full_header", "sequence", "Accession"]]
            # common = common.drop_duplicates(subset="full_header", keep="first")
            # # print(common)
            # right = common.merge(og_fasta, on="isolate", how="right")
            right = common
            print(right)
            
            right["full_header"] = right["full_header_y"].fillna(right["full_header_x"])
            right["sequence"] = right["sequence_x"]
            right = right[["full_header", "sequence"]]
            print(right)
            df_to_fasta(right, "", key[:-4] + "_" + str(count1) + str(count2) + "_edit_gisaid.fasta")
        else:
            df_to_fasta(og_fasta, "", key[:-4] + "_none_found_gisaid.fasta")
        count2 += 1
    count1 += 1

C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/Andersen_NCBI_Virus_GISAID/11-01-2021--07-25-2025_Antarctica_North_America_South_America/all_A3_HA_combined_11-01-2021--07-25-2025.fasta
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/Andersen_NCBI_Virus_GISAID/11-01-2021--07-25-2025_Antarctica_North_America_South_America/all_A3_MP_combined_11-01-2021--07-25-2025.fasta
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/Andersen_NCBI_Virus_GISAID/11-01-2021--07-25-2025_Antarctica_North_America_South_America/all_A3_NA_combined_11-01-2021--07-25-2025.fasta
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/Andersen_NCBI_Virus_GISAID/11-01-2021--07-25-2025_Antarctica_North_America_South_America/all_A3_NP_combined_11-01-2021--07-25-2025.fasta
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/Andersen_NCBI_Virus_GISAID/11-01-2021--07-25-2025_Antarctica_North_America_South_America/all_A3_NS_combined_11-01-202